# Step 4: Evaluation, Calibration, SHAP and Secondary Analyses (revision 1)

| Output | Content | Comments |
|---|---|---|
| Table 1 | Training vs. test characteristics with **SMD** (no hypothesis tests); patients with ≥1 event | R1-m2, R2-m2 |
| Outcome components | Frequency of each component of the composite outcome | R2-M2 |
| Table 2 | AUROC, AUPRC, Brier, calibration intercept/slope, threshold metrics with **patient-level cluster-bootstrap 95% CIs** | R1-M5, R2-M1, R1-M3(c), R2-M5 |
| Threshold table | Sensitivity / specificity / PPV / NPV at 10% and 20% predicted risk | R1-M6 |
| Fig. ROC | Three models overlaid in one panel | R1-m7 |
| Fig. calibration, DCA | Decile + LOWESS calibration plot; decision curve | R1-M6, R2-M4 |
| Fig. SHAP | LightGBM SHAP summary (descriptive, non-causal) | R2-M6 |
| Secondary analyses | Age / BSA vs. event: naive vs. **cluster-robust GLM / GEE** | R1-M5, R2-M1, R1-m4 |


In [ ]:
import sys, os, platform, json, warnings
sys.path.insert(0, os.path.abspath("../src"))   # revision-1 shared modules (src/fnpred)
import numpy as np, pandas as pd, sklearn, lightgbm, xgboost
print("python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__,
      "| scikit-learn", sklearn.__version__, "| lightgbm", lightgbm.__version__, "| xgboost", xgboost.__version__)
import matplotlib.pyplot as plt, joblib
from fnpred import evaluation as ev, stats as st

In [ ]:
DATA_DIR   = "../../data"
EXPERIMENT = "main"            # "main" | "temporal" | "no_gcsf"
MODEL_DIR  = f"{DATA_DIR}/step3_model_outputs/{EXPERIMENT}"
OUT_DIR    = f"{DATA_DIR}/step4_evaluation_outputs/{EXPERIMENT}"
N_BOOT     = 2000
os.makedirs(OUT_DIR, exist_ok=True)

df    = pd.read_parquet(f"{DATA_DIR}/step2_processing_outputs/step2_all_tagt_feat")
split = pd.read_parquet(f"{MODEL_DIR}/split.parquet")
pred  = pd.read_parquet(f"{MODEL_DIR}/pred_test.parquet")
cfg   = json.load(open(f"{MODEL_DIR}/config.json"))
thr   = json.load(open(f"{MODEL_DIR}/thresholds.json"))
feats = json.load(open(f"{MODEL_DIR}/features.json"))
TARGET = cfg["target"]
ALGOS  = [c for c in pred.columns if c not in ("hs", "DO_DATE", TARGET)]
df = df.merge(split, on=["hs", "DO_DATE"], how="left")
y, g = pred[TARGET].to_numpy().astype(int), pred["hs"].to_numpy()
ALGOS

## Table 1 (SMD) and patient-level event counts

In [ ]:
d = df[df["set"].isin(["train", "test"])].copy()
d["Female"] = d["PT_SEX"]                       # step 2 coding: M=0, F=1
pt = (d.groupby(["set", "hs"]).agg(AGE=("AGE", "first"), Female=("Female", "first"),
                                   any_event=(TARGET, "max"), n_days=(TARGET, "size")).reset_index())
summary = pd.DataFrame({
    s: {"patients": (pt.set == s).sum(),
        "chemotherapy administration days": (d.set == s).sum(),
        "event incidence (days)": f"{d.loc[d.set == s, TARGET].mean():.1%} "
                                  f"({int(d.loc[d.set == s, TARGET].sum()):,}/{(d.set == s).sum():,})",
        "patients with >=1 event": f"{int(pt.loc[pt.set == s, 'any_event'].sum()):,} "
                                   f"({pt.loc[pt.set == s, 'any_event'].mean():.1%})"}
    for s in ["train", "test"]})
t1 = st.table1_smd(pt[pt.set == "train"], pt[pt.set == "test"], ["AGE", "n_days"], ["Female", "any_event"])
t1d = st.table1_smd(d[d.set == "train"], d[d.set == "test"], [], [TARGET]).assign(level="administration day")
display(summary); display(pd.concat([t1.assign(level="patient"), t1d]))
summary.to_csv(f"{OUT_DIR}/table1_counts.csv"); pd.concat([t1.assign(level="patient"), t1d]).to_csv(f"{OUT_DIR}/table1_smd.csv", index=False)

## Outcome components (R2-M2)

In [ ]:
comp = [c for c in df.columns if c.startswith("Event_") or c in ("Valid_FN", "Therapeutic_GCSF_Event")]
if comp:
    tab = pd.DataFrame({c: [int(df[c].sum()), df[c].mean(), df.groupby("hs")[c].max().sum()] for c in [TARGET] + comp},
                       index=["administration days", "proportion of days", "patients with >=1"]).T
    display(tab); tab.to_csv(f"{OUT_DIR}/outcome_components.csv")
else:
    print("component columns not found (run the revised step 2)")

## Table 2: discrimination, calibration and threshold metrics with patient-level cluster-bootstrap 95% CIs

In [ ]:
ci = {}
for a in ALGOS:
    th = {k: v for k, v in thr[a].items() if k != "default_0.50" and np.isfinite(v)}
    ci[a] = ev.cluster_bootstrap(y, pred[a].to_numpy(), g, thresholds=th, n_boot=N_BOOT)
table2 = pd.DataFrame({a: ci[a].apply(ev.fmt_ci, axis=1) for a in ALGOS})
display(table2)
table2.to_csv(f"{OUT_DIR}/table2_metrics_ci.csv")
pd.concat(ci, names=["model", "metric"]).to_csv(f"{OUT_DIR}/table2_metrics_ci_long.csv")

In [ ]:
# Threshold table: all thresholds, point estimates and confusion counts
rows = []
for a in ALGOS:
    for name, t in thr[a].items():
        if np.isfinite(t):
            rows.append({"model": a, "threshold_rule": name, **ev.threshold_metrics(y, pred[a].to_numpy(), t)})
thr_table = pd.DataFrame(rows); display(thr_table.round(3))
thr_table.to_csv(f"{OUT_DIR}/threshold_metrics.csv", index=False)

## Figures: ROC (overlaid), calibration, decision curve

In [ ]:
RAW   = [a for a in ALGOS if not a.endswith("_recal")]
RECAL = [a for a in ALGOS if a.endswith("_recal")]
ax = ev.plot_roc_overlay(y, {a: pred[a].to_numpy() for a in RAW}, ci={a: ci[a] for a in RAW})
plt.savefig(f"{OUT_DIR}/Fig_ROC_overlay.tiff", dpi=300, bbox_inches="tight"); plt.show()
for grp, tag in [(RAW, "raw"), (RECAL, "recalibrated")]:
    if not grp: continue
    ev.plot_calibration(y, {a: pred[a].to_numpy() for a in grp})
    plt.savefig(f"{OUT_DIR}/Fig_calibration_{tag}.tiff", dpi=300, bbox_inches="tight"); plt.show()
    plt.figure(); ev.plot_decision_curve(y, {a: pred[a].to_numpy() for a in grp})
    plt.savefig(f"{OUT_DIR}/Fig_DCA_{tag}.tiff", dpi=300, bbox_inches="tight"); plt.show()
    pd.concat({a: ev.decision_curve(y, pred[a].to_numpy()) for a in grp}).to_csv(f"{OUT_DIR}/dca_{tag}.csv")

## SHAP (LightGBM) — descriptive contribution to predictions, not causal effects (R2-M6)

In [ ]:
import shap
pipe = joblib.load(f"{MODEL_DIR}/model_lgb.joblib")
X_test = df.loc[df["set"] == "test", feats]
Xt = pd.DataFrame(pipe[:-1].transform(X_test), columns=feats, index=X_test.index)
sv = shap.TreeExplainer(pipe[-1].booster_)(Xt)
if sv.values.ndim == 3:
    sv = sv[:, :, 1]
name_map = {"4291427": "Nivolumab", "4240403": "Etoposide", "DIAGNOSIS_C50": "Breast cancer (C50)",
            "lastWBC": "Latest WBC count", "DIAGNOSIS_C77": "Secondary malignant neoplasm of lymph nodes (C77)",
            "lastNEUT#": "Latest ANC", "BPH": "Systolic BP", "4291435": "Pembrolizumab",
            "MS_HEIGHT": "Height", "MS_WEIGHT": "Weight", "AGE": "Age", "TOTAL_GLAS_PREDOSE": "Prophylactic G-CSF dose"}
sv.feature_names = [name_map.get(n, n) for n in feats]
shap.plots.beeswarm(sv, max_display=11, show=False)
plt.savefig(f"{OUT_DIR}/Fig_SHAP_LightGBM.tiff", dpi=300, bbox_inches="tight"); plt.show()
pd.Series(np.abs(sv.values).mean(0), index=sv.feature_names).sort_values(ascending=False).head(30).to_csv(f"{OUT_DIR}/shap_mean_abs.csv")

## Secondary exploratory analyses: age / BSA and event incidence (R1-M5, R2-M1, R1-m4)
Whole analytic cohort, as in the initial submission. Patient-level clustering is handled by cluster-robust SEs and GEE (exchangeable).
The etoposide dose–age analysis (initial Section 3.5) has been removed (R1-m3).

In [ ]:
sa = df.copy()
sa["BSA"] = st.du_bois_bsa(sa["MS_HEIGHT"], sa["MS_WEIGHT"])
sec = pd.concat([st.clustered_logit(sa, TARGET, "AGE", "hs", per=1),
                 st.clustered_logit(sa, TARGET, "AGE", "hs", per=10),
                 st.clustered_logit(sa, TARGET, "BSA", "hs", per=1)])
display(sec.round(4)); sec.to_csv(f"{OUT_DIR}/secondary_age_bsa.csv", index=False)